# Autograd

*Easy · from [mlcode](https://mlcode-io.vercel.app/problems/torch-autograd)*

Get PyTorch to compute a derivative for you.

For `f(x) = 3x^2 + 2x`, return `df/dx` at each value in `x_values`, using
autograd rather than the derivative you can do in your head.

The mechanics:

```
x = torch.tensor(x_values, dtype=torch.float32, requires_grad=True)
y = 3 * x ** 2 + 2 * x
y.sum().backward()
x.grad
```

`requires_grad=True` is what makes torch record the operations. `backward()`
walks that recording in reverse and accumulates the result into `x.grad`.

Calling `backward()` needs a scalar, because a gradient is only defined
against one number. Summing first is the usual trick: each element of `x`
affects exactly one element of `y`, so `d(sum)/dx_i` is `dy_i/dx_i`, which is
what you wanted.

Return a tensor of shape `(n,)`. You have already written this derivative by
hand in the calculus problems; the point here is to see the same answer arrive
without you deriving it.

### Example

```
grad_of_quadratic([0.0, 1.0, 2.0])  ->  tensor([2., 8., 14.])
```

---

Write your solution in the cell below, then run the grading cell at the bottom. Your work is recorded against the same account as the site, so a pass here shows there.

In [ ]:
#@title Connect this notebook to mlcode { display-mode: "form" }
#@markdown Paste the code from the problem page, then run this cell.
CODE = ""  #@param {type:"string"}

SITE = "https://mlcode-io.vercel.app"
SLUG = "torch-autograd"

import json, urllib.request, urllib.error

def _get(path):
    with urllib.request.urlopen(SITE + path, timeout=30) as r:
        return r.read()

# The harness is fetched rather than pasted in, so the cases run here exactly
# as they would on the site.
for _name in ("harness.py", "notebook.py"):
    with open("/content/" + _name, "wb") as _f:
        _f.write(_get("/" + _name))

try:
    _meta = json.loads(_get(f"/api/colab?op=tests&code={CODE}&slug={SLUG}"))
    print("connected:", _meta["title"])
    print("torch:", __import__("torch").__version__)
except urllib.error.HTTPError as e:
    print("could not connect:", json.loads(e.read()).get("error", e.reason))
    print("Get a fresh code from the problem page; they last three hours.")


In [ ]:
import torch


def grad_of_quadratic(x_values):
    """df/dx for f(x) = 3x^2 + 2x, computed with autograd.

    x_values: list of floats
    returns: torch.Tensor of shape (n,)
    """
    pass

In [ ]:
#@title Grade this notebook { display-mode: "form" }
#@markdown Runs the hidden tests and records the attempt on the site.
import json, sys, time, urllib.request, urllib.error

sys.path.insert(0, "/content")
import harness

# The tests are fetched now rather than stored in the notebook, so a reader
# cannot read them before attempting the problem.
_tests = json.loads(_get(f"/api/colab?op=tests&code={CODE}&slug={SLUG}"))["tests"]

# `In` is the execution history, not the current cells: a wrong first attempt,
# a cell that errored and three re-runs are all still in it. Saving that would
# put something on the site that does not even replay. Rebuild the source from
# what is actually defined now instead, and keep the history only as a fallback
# for a solution written at module level.
def _collect_source():
    import ast, inspect, types

    # This notebook's own form cells are not part of anyone's answer.
    _cells = [c for c in In[1:] if not c.lstrip().startswith("#@title")]

    _imports, _seen = [], set()
    _consts = dict()
    for _cell in _cells:
        for _line in _cell.split("\n"):
            _t = _line.strip()
            if (_t.startswith("import ") or _t.startswith("from ")) and _t not in _seen:
                _seen.add(_t)
                _imports.append(_t)
        try:
            _tree = ast.parse(_cell)
        except SyntaxError:
            continue
        # Module level constants the functions below may close over. Last
        # assignment wins, because an edited cell is run again.
        for _node in _tree.body:
            if isinstance(_node, ast.Assign) and len(_node.targets) == 1:
                _target = _node.targets[0]
                if isinstance(_target, ast.Name):
                    _text = ast.get_source_segment(_cell, _node)
                    if _text and len(_text) <= 300:
                        _consts[_target.id] = _text

    _defs, _defined = [], set()
    for _name, _obj in list(globals().items()):
        if _name.startswith("_"):
            continue
        if isinstance(_obj, (types.FunctionType, type)) and getattr(_obj, "__module__", "") == "__main__":
            try:
                _lead = inspect.getcomments(_obj) or ""
                _defs.append(_lead + inspect.getsource(_obj).rstrip())
                _defined.add(_name)
            except (OSError, TypeError):
                pass

    # Nothing defined means the solution is not shaped like a function, so fall
    # back to what was typed rather than returning an empty file.
    if not _defs:
        return "\n\n".join(_cells)

    _keep = []
    for _name, _text in _consts.items():
        if _name in _defined or _name in ("CODE", "SITE", "SLUG"):
            continue
        # Only values small enough to be worth carrying, not a loaded dataset.
        if isinstance(globals().get(_name), (int, float, complex, bool, str, bytes, tuple, list, dict, set, type(None))):
            _keep.append(_text)

    _blocks = ["\n".join(_imports), "\n".join(_keep), "\n\n\n".join(_defs)]
    return "\n\n\n".join(b for b in _blocks if b.strip()) + "\n"

_source = _collect_source()

_started = time.time()
_results = harness.run_tests(_tests, dict(globals()), 60, _source, "submit")
_elapsed = int((time.time() - _started) * 1000)

_passed = sum(1 for r in _results if r["passed"])
for r in _results:
    print(("  pass  " if r["passed"] else "  FAIL  ") + r["name"])
    if not r["passed"] and r["message"]:
        print("        " + r["message"].replace("\n", "\n        "))
print(f"\n{_passed}/{len(_results)} passed")

_body = json.dumps({
    "code": CODE, "slug": SLUG, "passedCount": _passed,
    "totalCount": len(_results), "runtimeMs": _elapsed, "source": _source,
}).encode()
_req = urllib.request.Request(SITE + "/api/colab", data=_body,
                              headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(_req, timeout=30) as r:
        print("recorded on mlcode:", "solved" if json.load(r)["passed"] else "attempt saved")
except urllib.error.HTTPError as e:
    print("could not record:", json.loads(e.read()).get("error", e.reason))
